In [ ]:
#| include: false

!pip install garminconnect pandas matplotlib folium
from garminconnect import Garmin
import getpass

email = input("Garmin email: ")
password = getpass.getpass("Garmin wachtwoord: ") 

try:
    client = Garmin(email, password)
    client.login()
    print("Succesvol ingelogd!")
    print(f"Welkom, {client.get_full_name()}")
except Exception as e:
    print(f"Oeps, inloggen mislukt: {e}")

In [ ]:
import pandas as pd

activities = client.get_activities(0, 582)
df_raw = pd.DataFrame(activities)

relevante_kolommen = [
    'startTimeLocal', 
    'distance', 
    'duration', 
    'averageHR', 
    'maxHR' 
]

bestaande_kolommen = [col for col in relevante_kolommen if col in df_raw.columns]
df_raw[bestaande_kolommen].head()

In [ ]:
df_raw['afstand_km'] = df_raw['distance'] / 1000
df_raw['duur_min'] = df_raw['duration'] / 60

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import datetime

if 'datum' not in df_raw.columns:
    df_raw['datum'] = pd.to_datetime(df_raw['startTimeLocal'], utc=True).dt.tz_localize(None)

if 'afstand_km' not in df_raw.columns:
    df_raw['afstand_km'] = df_raw['distance'] / 1000

huidig_jaar = datetime.datetime.now().year
start_van_het_jaar = pd.to_datetime(f"{huidig_jaar}-01-01")

df_recent = df_raw[df_raw['datum'] >= start_van_het_jaar].copy()
df_week = df_recent.groupby(pd.Grouper(key='datum', freq='W-MON'))['afstand_km'].sum().reset_index()

plt.figure(figsize=(12, 6))
plt.bar(df_week['datum'], df_week['afstand_km'], width=5, color='#2c3e50', label='Afgelegde kilometers')
plt.title(f'Wekelijkse Kilometers ({huidig_jaar})', fontsize=16, fontweight='bold')
plt.xlabel('Datum', fontsize=12)
plt.ylabel('Totale Afstand (km)', fontsize=12)
plt.legend()
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.show()

In [ ]:
if 'sport' not in df_raw.columns and 'activityType' in df_raw.columns:
    df_raw['sport'] = df_raw['activityType'].apply(lambda x: x.get('typeKey', '') if isinstance(x, dict) else str(x))

huidig_jaar = datetime.datetime.now().year
start_van_het_jaar = pd.to_datetime(f"{huidig_jaar}-01-01")
toegestane_sporten = ['running', 'trail_running']

df_recent = df_raw[
    (df_raw['datum'] >= start_van_het_jaar) & 
    (df_raw['sport'].isin(toegestane_sporten))
].copy()

df_week = df_recent.groupby(pd.Grouper(key='datum', freq='W-MON'))['afstand_km'].sum().reset_index()

plt.figure(figsize=(12, 6))
plt.bar(df_week['datum'], df_week['afstand_km'], width=5, color='#2c3e50', label='Hardloop & Trail km')
plt.title(f'Wekelijkse Hardloop Kilometers ({huidig_jaar})', fontsize=16, fontweight='bold')
plt.xlabel('Datum', fontsize=12)
plt.ylabel('Totale Afstand (km)', fontsize=12)
plt.legend()
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.show()

In [ ]:
df_pace = df_raw[
    (df_raw['datum'] >= start_van_het_jaar) & 
    (df_raw['sport'].isin(toegestane_sporten)) &
    (df_raw['afstand_km'] > 0)
].copy()

df_pace['tempo_decimaal'] = df_pace['duur_min'] / df_pace['afstand_km']
df_pace = df_pace.sort_values('datum')
df_pace['trendlijn'] = df_pace['tempo_decimaal'].rolling(window=5, min_periods=1).mean()

plt.figure(figsize=(12, 6))
plt.scatter(df_pace['datum'], df_pace['tempo_decimaal'], color='#3498db', alpha=0.6, label='Individuele Run')
plt.plot(df_pace['datum'], df_pace['trendlijn'], color='#e74c3c', linewidth=2, label='Trend (Rollend gem. 5 runs)')
plt.title(f'Tempo per Run ({huidig_jaar})', fontsize=16, fontweight='bold')
plt.xlabel('Datum', fontsize=12)
plt.ylabel('Tempo (min/km, decimaal)', fontsize=12)
plt.gca().invert_yaxis()
plt.legend()
plt.grid(axis='both', linestyle='--', alpha=0.4)
plt.savefig('tempo-thumbnail.png', dpi=100, bbox_inches='tight')
plt.show()

In [ ]:
df_hartslag = df_raw[
    (df_raw['datum'] >= start_van_het_jaar) & 
    (df_raw['sport'].isin(toegestane_sporten)) &
    (df_raw['averageHR'] > 0) 
].copy()

df_hartslag = df_hartslag.sort_values('datum')
df_hartslag['trendlijn'] = df_hartslag['averageHR'].rolling(window=5, min_periods=1).mean()

plt.figure(figsize=(12, 6))
plt.scatter(df_hartslag['datum'], df_hartslag['averageHR'], color='#9b59b6', alpha=0.5, label='Gem. Hartslag per Run')
plt.plot(df_hartslag['datum'], df_hartslag['trendlijn'], color='#8e44ad', linewidth=2.5, label='Trend (5 runs)')
plt.title(f'Gemiddelde Hartslag per Run ({huidig_jaar})', fontsize=16, fontweight='bold')
plt.xlabel('Datum', fontsize=12)
plt.ylabel('Hartslag (bpm)', fontsize=12)
plt.legend()
plt.grid(axis='y', linestyle='--', alpha=0.6)
plt.show()

In [ ]:
df_combo = df_raw[
    (df_raw['datum'] >= start_van_het_jaar) & 
    (df_raw['sport'].isin(toegestane_sporten)) &
    (df_raw['averageHR'] > 0) &
    (df_raw['afstand_km'] > 0)
].copy()

df_combo['tempo_decimaal'] = df_combo['duur_min'] / df_combo['afstand_km']
df_combo = df_combo.sort_values('datum')
df_combo['trend_hr'] = df_combo['averageHR'].rolling(window=5, min_periods=1).mean()
df_combo['trend_tempo'] = df_combo['tempo_decimaal'].rolling(window=5, min_periods=1).mean()

fig, ax1 = plt.subplots(figsize=(12, 6))
kleur_hr = '#e74c3c'  
ax1.scatter(df_combo['datum'], df_combo['averageHR'], color=kleur_hr, alpha=0.2)
ax1.plot(df_combo['datum'], df_combo['trend_hr'], color=kleur_hr, linewidth=3, label='Trend Hartslag (bpm)')
ax1.set_xlabel('Datum', fontsize=12)
ax1.set_ylabel('Hartslag (bpm)', color=kleur_hr, fontsize=12, fontweight='bold')
ax1.tick_params(axis='y', labelcolor=kleur_hr)

ax2 = ax1.twinx()  
kleur_tempo = '#3498db'  
ax2.scatter(df_combo['datum'], df_combo['tempo_decimaal'], color=kleur_tempo, alpha=0.2)
ax2.plot(df_combo['datum'], df_combo['trend_tempo'], color=kleur_tempo, linewidth=3, label='Trend Tempo (min/km)')
ax2.set_ylabel('Tempo (min/km)', color=kleur_tempo, fontsize=12, fontweight='bold')
ax2.tick_params(axis='y', labelcolor=kleur_tempo)
ax2.invert_yaxis()

plt.title(f'Hartslag vs Tempo', fontsize=16, fontweight='bold')
ax1.grid(axis='x', linestyle='--', alpha=0.4)
fig.legend(loc='upper right', bbox_to_anchor=(0.9, 0.88))
plt.show()

In [ ]:
import folium
import warnings
warnings.filterwarnings('ignore')

print("Kaarten genereren...")
m = folium.Map(location=[52.46, 4.62], zoom_start=11, tiles='CartoDB positron')

df_hardloop = df_raw[
    (df_raw['datum'] >= start_van_het_jaar) & 
    (df_raw['sport'].isin(toegestane_sporten)) &
    (df_raw['afstand_km'] > 0)
].copy()

if 'gpsPoints' in df_raw.columns:
    total_runs = len(df_hardloop)
    ignored = 0
    
    for i, (idx, run) in enumerate(df_hardloop.iterrows()):
        if (i + 1) % 15 == 0:
            print(f"Voortgang: {i + 1} van de {total_runs} runs verwerkt...")
        
        gps_points = run['gpsPoints']
        if gps_points and len(gps_points) > 0:
            coords = [[point['lat'], point['lon']] for point in gps_points]
            folium.PolyLine(coords, color='#fc4c02', weight=2, opacity=0.8).add_to(m)
        else:
            ignored += 1
    
    print(f"Klaar! {total_runs - ignored} routes getekend. {ignored} runs genegeerd.")
    m.show()